# Visualizing Data in Roman ASDF Files with `jdaviz`

***

## Kernel Information and Read-Only Status

To run this notebook, please select the "Roman Research Nexus {VERSION}"  kernel at the top right of your window. For example, "Roman Research Nexus 2026.1".

This notebook is read-only. You can run cells and make edits, but you must save changes to a different location. We recommend saving the notebook within your home directory, or to a new folder within your home (e.g. <span style="font-variant:small-caps;">file > save notebook as > my-nbs/nb.ipynb</span>). Note that a directory must exist before you attempt to add a notebook to it.

## Introduction
This tutorial demonstrates how to visualize and explore Roman WFI image data arrays by using the Jdaviz tool to interactively explore WFI image data. A companion tutorial shows [create static images with world coordinate system (WCS) in `matplotlib`](LINK-TO-COMPANION-NOTEBOOK-IF-DESIRED).

We focus on how to visualize WFI Level 2 (L2; calibrated rate image) data in ASDF format. For WFI, L2 indicates that the data have been processed to flag and/or correct for detector-level effects (e.g., saturation, classic non-linearity, etc.) and the resultants fitted into a count rate image. Each L2 ASDF file contains a single WFI detector, thus a complete WFI exposure is made up of 18 L2 files. For more information WFI L2 files, please see the RDox article on [Data Levels and Products](https://roman-docs.stsci.edu/data-handbook-home/wfi-data-format/data-levels-and-products#DataLevelsandProducts-Level2-CalibratedExposuresLevel2).

## Imports
- *astropy.visualization.simple_norm* for automatically scaling image arrays
- *astropy.coordinates.SkyCoord* to create Python objects containing sky coordinate transforms
- *astropy.table.Table* to create Astropy tables
- *numpy* for array calculations and manipulation
- *jdaviz* to examine Wide Field Instrument (WFI) images interactively
- *roman_datamodels* for opening Roman WFI ASDF files
- *s3fs* to access data in an AWS S3 bucket

In [ ]:
%matplotlib inline
from astropy.visualization import simple_norm
from astropy.coordinates import SkyCoord
from astropy.table import Table
import numpy as np

import jdaviz as jd
import roman_datamodels as rdm
import s3fs

## Interactively Examine the Data Using jdaviz

### Introduction and Setup

We will use jdaviz, an interactive data visualization and analysis package for Jupyter notebooks, to explore the 2-D image arrays contained within WFI L2 ASDF files. We highly recommend that users consult the [jdaviz documentation](https://jdaviz.readthedocs.io/en/stable/), which describes many of the features in jdaviz in detail. In this tutorial, we will cover the basics to get you started.

The first cell below will open jdaviz.

In [ ]:
# open jdaviz in a sidecar at the bottom of this jupyter lab workspace:
jd.show('sidecar:split-bottom', title='jdaviz')

### Streaming Data

Next, we load a Roman WFI image into jdaviz. We use the same data file in both this notebook and [the static `matplotlib` notebook](ANOTHER-LINK-TO-POTENTIALLY-FILL-IN). 

jdaviz only streams the `data` array in the viewer by default. Additional arrays (for example., the DQ array) may be loaded using the `extension` optional argument. 

An example demonstrating how to load the data quality array is provided in a commented line in the following cell. For more information on the arrays contained within WFI L2 files, please see the RDox article on WFI [Data Levels and Products](https://roman-docs.stsci.edu/data-handbook-home/wfi-data-format/data-levels-and-products#DataLevelsandProducts-L2ScienceDataSpecifications).

In [ ]:
s3_bucket = "s3://stpubdata/roman/nexus/soc_simulations/tutorial_data/roman-2026.1/"

file_system = s3fs.S3FileSystem(anon=True)

image_1_prefix = s3_bucket + 'r0003201001001001004_0001_wfi01_f106_cal.asdf'
image_file_1 = file_system.open(image_1_prefix, 'rb')
position_1 = rdm.open(image_file_1)
wcs1 = position_1.meta.wcs

### Programmatically Adjusting Display

Next, we load the image into jdaviz. jdaviz loads the `data` array in the viewer by default. Additional arrays (e.g., like the data quality array array) may be loaded using the `extension` optional argument. An example demonstrating how to load the data quality array is provided in a commented line in the following cell. For more information on the arrays contained within WFI L2 files, please see the RDox article on WFI [Data Levels and Products](https://roman-docs.stsci.edu/data-handbook-home/wfi-data-format/data-levels-and-products#DataLevelsandProducts-L2ScienceDataSpecifications).

In [ ]:
# load the image at "position 1" into jdaviz:
jd.load(position_1, data_label='WFI01_POS1', format='Image')

# alternatively, one could load the DATA and DQ extensions like so:
# jd.load(position_1, extension=['data', 'dq'], data_label='WFI01 DQ')

The colormap, stretch, and scale limits may be adjusted interactively in the viewer by clicking the settings (cog) icon towards the upper left of jdaviz (tooltip labeled 'Settings and options'), and selecting the "Plot Options" tab beneath the settings icon. Note that there are additional options such as the minimum and maximum scale limits under the "More Stretch Options" expander.

If we know the settings we want to apply, we can do so via the jdaviz API as follows:

In [ ]:
plot_options = jd.plugins['Plot Options']

# we'll adjust the plot options for the image with the following `data_label`:
plot_options.layer = 'WFI01_POS1'

# Apply the viridis colormap with an Arcsinh stretch between (0.5, 4) counts:
plot_options.image_colormap = 'Viridis'
plot_options.stretch_function = 'arcsinh'
plot_options.stretch_vmin = 0.5
plot_options.stretch_vmax = 4

Now we will display a second image, showing the same field but slightly dithered, and we will link the two images by their WCS. The second image is dithered by ~200 arcseconds compared to the first, so sources should move by ~1000 pixels between the two images.

In [ ]:
image_2_prefix = s3_bucket + 'r0003201001001001004_0002_wfi01_f106_cal.asdf'
image_file_2 = file_system.open(image_2_prefix, 'rb')
position_2 = rdm.open(image_file_2)
wcs2 = position_2.meta.wcs

jd.load(position_2, data_label='WFI01_POS2', format='Image')

In [ ]:
# now adjust the plot options for the image with the following `data_label`:
plot_options.layer = 'WFI01_POS2'

# Apply the same plot options as the first image:
plot_options.image_colormap = 'Viridis'
plot_options.stretch_function = 'arcsinh'
plot_options.stretch_vmin = 0.5
plot_options.stretch_vmax = 4

Now that the second image has been displayed, we can blink between the two using the "b" button. Make sure that your cursor is placed over the image to make it the active window, then blink between the images. Try identifying a few sources that are common between the two images to see the effect of dithering between exposures.

Similarly to [the matplotlib notebook](ADD-ANOTHER-LINK-IF-NEEDED), let's investigate a region of interest and focus on the extended source in the right portion of the first image. In this case, we know the galaxy has science pixel coordinates of (X, Y) ~ (3480, 3030) pixels in the first image. We can use the WCS from the first image (which we saved as the variable `wcs1`) to transform this to sky coordinates:

In [ ]:
ra, dec = wcs1(3480, 3030)
print(f'RA = {ra} deg, Dec = {dec} deg')

Next, we center the viewer at this position. We can use either the sky coordinates, or alternatively the pixel coordinates (which have been commented out). Note that the centering occurs on the currently displayed image, so if you have the second image actively displayed and use the pixel coordinates, it will center to the incorrect location when using pixel coordinates from the first image. Using sky coordinates is independent of which image is actively displayed.

In [ ]:
# Center the image on given sky coordinates.
sky = SkyCoord(ra=ra, dec=dec, unit='deg')
viewer = jd.viewers['Image']
viewer.center_on(sky)

# Center the image on given pixel coordinates.
# viewer.center_on((3480, 3030))

We can also set the zoom level to better display the region around the extended source. The zoom level settings are such as:

* 1: real-pixel-size.
* 2: zoomed in by a factor of 2.
* 0.5: zoomed out by a factor of 2.
* 'fit' means zoomed to fit the whole image into display.

In this case, we will set the zoom level to 1.2 so we can better see the extended source.

Recall that to blink the images, you need to place the cursor over the viewer and press the "b" key on your keyboard. As you can see, when blinking the images in detector coordinates, the extended source is only visible in one image due to dithering between the exposures. Next, we will link the images by their WCS information so that the sources remain fixed in the display. Linking the images by WCS will reset the center and zoom, so let's reapply that as well.

**Important note:** Roman WFI ASDF files use a Generalized World Coordinate System (GWCS) object in Python to store the WCS transformation. The transformation is only well-defined within a bounding box, and moving outside that bounding box produces unexpected behavior, particularly due to the polynomial terms in the transformation between pixel and sky coordinates.

In [ ]:
orientation = jd.plugins['Orientation']
orientation.align_by = 'WCS'

viewer.zoom_level = 1000
viewer.center_on(sky)

We can pan and zoom manually to locate our galaxy of interest. With the cursor over the image, press the `b` key to blink between the two images. Blinking between the two images in any region where they overlap on the sky shows that we have succesfully aligned the images using sky coordinates.

You can also save the current view to a PNG file on your Nexus storage:

In [ ]:
viewer.save('my_image.png')

This will save the file to your current working directory on the Nexus. If you want to download this file, you can do so by right-clicking on the file in the file browser and selecting the "Download" option.

### Overlaying Catalog Data

A common workflow is to mark sources of interest on an image in jdaviz. First, let's read in the catalog used to generate this image using Roman I-Sim:

In [ ]:
# Read in catalog data from S3
cat_uri = s3_bucket + 'full_catalog.ecsv'

with file_system.open(cat_uri, 'rb') as catalog_file:
    tab = Table.read(catalog_file, format='ascii.ecsv')

There are a lot of sources in this file, but let's pare them down somewhat for display purposes. In this case, let's filter down to the brightest ($m_{AB}$ < 18) sources in the F106 filter. The flux columns in the table are in units of maggies, which may be converted to AB magnitudes as $m_{AB} = -2.5\log_{10}(f)$ where $m_{AB}$ is the magnitude in AB mags and $f$ is the flux in maggies.

Let's also filter the catalog for sources that lie in the overlap region between the two images. We can do this by transforming our sky positions to pixel positions with the WCS objects from each image and creating a mask by combining multiple conditions on the pixel positions (i.e., that they must lie within the bounds 0 <= X <= 4088 and 0 <= Y <= 4088).

In [ ]:
# Filter objects by brightness
bright = np.where(-2.5 * np.log10(tab['F106']) < 18)

# Make SkyCoord objects
coords = SkyCoord(ra = tab['ra'][bright].value, dec = tab['dec'][bright].value, unit = 'deg')

# Filter the SkyCoord objects on the WCS objects for each image to find only things in the overlap region.
x1, y1 = wcs1.world_to_pixel(coords)
x2, y2 = wcs2.world_to_pixel(coords)
mask = (x1 > 0) & (x1 < 4088) & (y1 > 0) & (y1 < 4088) & (x2 > 0) & (x2 < 4088) & (y2 > 0) & (y2 < 4088)

final_coords = Table({'coord': coords[mask]})
print(f'Number of sources in overlap region: {len(final_coords)}')

Now let's set up the markers and add them to the viewer:

In [ ]:
# Set up viewer marker parameters
viewer.marker = {'color': 'white', 'alpha': 0.8, 'markersize': 120, 'fill': False}

# Overlay markers
viewer.add_markers(final_coords, use_skycoord=True, marker_name='My_Markers')

We can also remove them if we don't want them any longer, or if we want to replace them with other markers:

In [ ]:
# Remove only My_Markers
viewer.remove_markers(marker_name='My_Markers')

# Remove ALL markers
# viewer.reset_markers()

For more advanced use cases such as interactive aperture photometry or analysis of line profiles, please consult the [jdaviz documentation](https://jdaviz.readthedocs.io/en/latest/).

***

## Additional Resources

- [jdaviz Documentation](https://jdaviz.readthedocs.io/en/stable/)
- [RDox WFI Data Levels and Products](https://roman-docs.stsci.edu/data-handbook-home/wfi-data-format/data-levels-and-products#DataLevelsandProducts-L2ScienceDataSpecifications)

## About this Notebook
**Author:** Tyler Desjardins, Brett Morris, R. Diaz  
**Updated On:** 2026-07-14

<table width="100%" style="border:none; border-collapse:collapse;">
  <tr style="border:none;">
    <td style="border:none; width:180px; white-space:nowrap;">
       <a href="#top" style="text-decoration:none; color:#0066cc;"> Top of page</a>
    </td>
    <td style="border:none; text-align:center;">
        <img src="https://raw.githubusercontent.com/spacetelescope/roman_notebooks/refs/heads/main/roman_logo.png" alt="roman_logo" width="50px">
    </td>
    <td style="border:none; text-align:right;">
       <img src="https://raw.githubusercontent.com/spacetelescope/roman_notebooks/refs/heads/main/stsci_logo2.png" width="90">
    </td>
  </tr>
</table>